In [87]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of records
num_records = 5000  # Change as needed

# Date range
start_date = datetime(2023, 4, 1)
end_date = datetime(2024, 3, 31)

# Hospital details
hospitals = [
    ("Toronto General", "Toronto Central LHIN (Former)"),
    ("Ottawa Civic", "Champlain LHIN (Former)"),
    ("Hamilton General", "Hamilton Niagara Haldimand Brant LHIN (Former)"),
    ("Sunnybrook", "Toronto Central LHIN (Former)"),
    ("Kingston General", "South East LHIN (Former)"),
    ("London Health Sciences", "South West LHIN (Former)"),
    ("Michael Garron", "Toronto Central LHIN (Former)"),
    ("Thunder Bay Regional", "North West LHIN (Former)")
]

# CTAS Levels and Their Weights (from real ED data)
ctas_levels = ["CTAS I-III (Discharged)", "CTAS IV-V (Discharged)", "Admitted"]
ctas_weights = [0.628, 0.248, 0.124]  # Normalized so they sum to 1


# Function to generate wait times (in hours) based on CTAS level
def generate_wait_time(ctas):
    """ Assigns wait times (in hours) using gamma distribution for realistic skew. """
    if ctas == "CTAS I-III (Discharged)":
        return round(np.random.gamma(shape=2.5, scale=1.44), 1)  # Median ~3.6h, 90th ~8.1h
    elif ctas == "CTAS IV-V (Discharged)":
        return round(np.random.gamma(shape=2.5, scale=0.88), 1)  # Median ~2.2h, 90th ~5.7h
    else:  # Admitted cases
        return round(np.random.gamma(shape=2.5, scale=5.72), 1)  # Median ~14.3h, 90th ~43.7h

# Generate Data
data = []
for _ in range(num_records):
    hospital, region = hospitals[np.random.randint(len(hospitals))]
    date_time = start_date + timedelta(days=np.random.randint((end_date - start_date).days))
    #date_time += timedelta(hours=np.random.randint(24), minutes=np.random.randint(60))
    
    age = np.random.normal(45, 20) # Mean age 45, std dev 20
    age = int(max(0, min(100, age)))  # Age range capped at 0-100
    
    ctas = np.random.choice(ctas_levels, p=ctas_weights)
    wait_time = generate_wait_time(ctas)  # Wait times in hours
    
    occupancy = np.random.randint(10, 200)
    
    season = "Winter" if date_time.month in [12, 1, 2] else \
             "Spring" if date_time.month in [3, 4, 5] else \
             "Summer" if date_time.month in [6, 7, 8] else "Fall"
    
    day_of_week = date_time.strftime("%A")
    
    
    data.append([hospital, region, date_time, age, ctas, wait_time, occupancy, day_of_week, season])

# Create DataFrame
columns = ["Hospital", "Region", "Date", "Age", "CTAS_Level", "WaitTime_Hours", 
           "ER_Occupancy", "DayOfWeek", "Season"]
df = pd.DataFrame(data, columns=columns)

# Save as CSV
df.to_csv("er_wait_times_ontario_updated_region.csv", index=False)

print("Dataset saved as 'er_wait_times_ontario_updated_region.csv'.")


Dataset saved as 'er_wait_times_ontario_updated_region.csv'.


In [163]:
# Load the dataset 
df = pd.read_csv("er_wait_times_ontario_updated_region.csv")

# Convert DateTime column to datetime format
df["Date"] = pd.to_datetime(df["Date"])


In [164]:
df

,Hospital,Region,Date,Age,CTAS_Level,WaitTime_Hours,ER_Occupancy,DayOfWeek,Season
0,Michael Garron,Toronto Central LHIN (Former),2024-03-14,55,CTAS I-III (Discharged),2.0,84,Thursday,Spring
1,Hamilton General,Hamilton Niagara Haldimand Brant LHIN (Former),2023-06-27,27,CTAS IV-V (Discharged),1.4,97,Tuesday,Summer
2,Sunnybrook,Toronto Central LHIN (Former),2024-01-19,74,CTAS I-III (Discharged),7.6,68,Friday,Winter
3,Michael Garron,Toronto Central LHIN (Former),2023-09-17,39,CTAS I-III (Discharged),2.8,73,Sunday,Fall
4,Toronto General,Toronto Central LHIN (Former),2023-08-09,53,CTAS I-III (Discharged),5.2,98,Wednesday,Summer
...,...,...,...,...,...,...,...,...,...
4995,Kingston General,South East LHIN (Former),2023-11-03,74,CTAS I-III (Discharged),7.8,10,Friday,Fall
4996,Thunder Bay Regional,North West LHIN (Former),2024-03-30,31,CTAS I-III (Discharged),1.9,83,Saturday,Spring
4997,Ottawa Civic,Champlain LHIN (Former),2023-05-29,56,CTAS I-III (Discharged),1.2,120,Monday,Spring
4998,Kingston General,South East LHIN (Former),2024-01-31,49,CTAS IV-V (Discharged),1.9,151,Wednesday,Winter


In [165]:
# Columns for the stages
stages = ['TriageTime', 'RegistrationTime', 'InitialAssessmentTime',
          'DiagnosticTime', 'TreatmentTime', 'DischargeTime']

# Function to generate Dirichlet-based proportions based on CTAS_Level
def generate_stage_times(row):
    ctas = row['CTAS_Level']
    total_time = row['WaitTime_Hours']  # Already in hours

    # Choose Dirichlet alphas based on CTAS level
    if ctas == "CTAS I-III (Discharged)":
        proportions = np.random.dirichlet([1.5, 1, 1.5, 1, 2, 1])
    elif ctas == "CTAS IV-V (Discharged)":
        proportions = np.random.dirichlet([1, 2, 1, 0.5, 1, 0.5])
    else:  # Admitted (if applicable in future)
        proportions = np.random.dirichlet([1.5, 1, 2, 2, 3, 1.5])

    # Scale by total wait time (in hours)
    return proportions * total_time

# Apply the function row-wise
stage_times = df.apply(generate_stage_times, axis=1, result_type='expand')
stage_times.columns = stages

# Add to original DataFrame
df = pd.concat([df, stage_times], axis=1)

# Optional: Round for cleaner display
df[stages] = df[stages].round(2)

# Preview result
df.head()

,Hospital,Region,Date,Age,CTAS_Level,WaitTime_Hours,ER_Occupancy,DayOfWeek,Season,TriageTime,RegistrationTime,InitialAssessmentTime,DiagnosticTime,TreatmentTime,DischargeTime
0,Michael Garron,Toronto Central LHIN (Former),2024-03-14,55,CTAS I-III (Discharged),2.0,84,Thursday,Spring,0.43,0.06,0.76,0.24,0.10,0.42
1,Hamilton General,Hamilton Niagara Haldimand Brant LHIN (Former),2023-06-27,27,CTAS IV-V (Discharged),1.4,97,Tuesday,Summer,0.14,0.32,0.30,0.13,0.52,0.00
2,Sunnybrook,Toronto Central LHIN (Former),2024-01-19,74,CTAS I-III (Discharged),7.6,68,Friday,Winter,2.40,0.50,3.99,0.13,0.23,0.36
3,Michael Garron,Toronto Central LHIN (Former),2023-09-17,39,CTAS I-III (Discharged),2.8,73,Sunday,Fall,0.59,0.55,0.36,0.32,0.89,0.08
4,Toronto General,Toronto Central LHIN (Former),2023-08-09,53,CTAS I-III (Discharged),5.2,98,Wednesday,Summer,0.18,0.79,1.05,0.85,1.45,0.87


In [166]:
# Save the updated DataFrame with stage times
df.to_csv("er_wait_times_ontario_updated_region_with_stages.csv", index=False)